In [1]:
import pandas as pd
import numpy as np

In [2]:
df = pd.read_csv('IIoT_Smart_Parking_Management.csv')

In [3]:
df = df[['Vehicle_Type_Weight', 'User_Type', 'Electric_Vehicle', 'Parking_Duration']]

df['Parking_Duration'] = df['Parking_Duration'].replace(0, 0.5)

n = len(df)
indices = np.random.permutation(n)

n_morning = int(round(n * 0.10))
n_midday = int(round(n * 0.60))
n_evening = n - n_morning - n_midday

morning_idx = indices[:n_morning]
midday_idx = indices[n_morning:n_morning + n_midday]
evening_idx = indices[n_morning + n_midday:]

def assign_arrival_times(target_idx, band_start, band_end):
    # Longer stays are scheduled earlier to keep departures before 23:00.
    day_minutes = 24 * 60
    durations = df.loc[target_idx, 'Parking_Duration']
    order = durations.sort_values(ascending=False).index
    times = {}
    current_time = band_start
    for row_idx in order:
        duration = df.at[row_idx, 'Parking_Duration']
        latest_arrival = min(band_end, (23 * 60) - (duration * 60))
        time_of_day = current_time % day_minutes
        if time_of_day > latest_arrival or time_of_day < band_start:
            current_time = (current_time // day_minutes + 1) * day_minutes + band_start
        times[row_idx] = current_time % day_minutes
        current_time += np.random.randint(1, 6)
    return times

arrival_map = {}
arrival_map.update(assign_arrival_times(morning_idx, 7 * 60, 11 * 60))
arrival_map.update(assign_arrival_times(midday_idx, 11 * 60, 17 * 60))
arrival_map.update(assign_arrival_times(evening_idx, 17 * 60, 23 * 60))

df['Arrival_Time'] = pd.Series(arrival_map)

df['Departure_Time'] = df['Arrival_Time'] + (df['Parking_Duration']*60)

df = df[df['Departure_Time'] <= 1440]

df = df.sort_values('Arrival_Time').reset_index(drop=True)

df['Vehicle_ID'] = np.arange(1, len(df) + 1)

def map_vehicle_type(weight):
    if weight <= 1200:
        return 'Compact'
    elif weight <= 1800:
        return 'Standard'
    else:
        return 'Large'
df['Vehicle_Type'] = df['Vehicle_Type_Weight'].apply(map_vehicle_type)

def map_user_type(user_type, duration):
    if user_type == 'Staff' or duration >= 7:
        return 'Staff'
    else:
        return 'Visitor'
df['User_Type'] = df.apply(lambda row: map_user_type(row['User_Type'], row['Parking_Duration']), axis=1)

df = df.drop(columns=['Vehicle_Type_Weight'])
df = df[['Vehicle_ID', 'Vehicle_Type', 'Arrival_Time', 'Parking_Duration', 'Departure_Time', 'User_Type', 'Electric_Vehicle']]

df.info()

df.to_csv('vehicles.csv', index=False)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 7 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   Vehicle_ID        1000 non-null   int64  
 1   Vehicle_Type      1000 non-null   object 
 2   Arrival_Time      1000 non-null   int64  
 3   Parking_Duration  1000 non-null   float64
 4   Departure_Time    1000 non-null   float64
 5   User_Type         1000 non-null   object 
 6   Electric_Vehicle  1000 non-null   int64  
dtypes: float64(2), int64(3), object(2)
memory usage: 54.8+ KB


# vehicles.csv Dataset Guide

This dataset is a compact, vehicle-level view derived from the original IIoT smart parking data. It is designed to feed scheduling, allocation, and optimization models with clean, consistent vehicle arrival and departure information.

Features are made to simulate the non-residential parkings such as malls, parks, factories ...

## How the dataset was produced
- Source fields: Vehicle_Type_Weight, User_Type, Electric_Vehicle, Parking_Duration.
- Synthetic arrival times are assigned so that:
  - 10% of arrivals are between 07:00-11:00.
  - 60% of arrivals are between 11:00-17:00.
  - 30% of arrivals are between 17:00-23:00.
- Inter-arrival spacing is kept between 1-5 minutes within each time band.
- Longer parking durations are scheduled earlier within a band to keep departures before 23:00.
- Departure_Time is computed as Arrival_Time + (Parking_Duration * 60).
- Vehicle_Type is derived from Vehicle_Type_Weight: Compact (<=1200), Standard (<=1800), Large (>1800).
- User_Type is normalized: Staff if original User_Type is Staff or Parking_Duration >= 7, otherwise Visitor.

## Columns
- Vehicle_ID: Sequential ID for each record.
- Vehicle_Type: Compact, Standard, or Large.
- Arrival_Time: Minutes after midnight (0-1439).
- Parking_Duration: Duration in hours (0.5 minimum).
- Departure_Time: Minutes after midnight, computed from Arrival_Time and Parking_Duration.
- User_Type: Staff or Visitor.
- Electric_Vehicle: 0/1 indicator from source.

## Notes and usage
- Arrival_Time and Departure_Time are minute-of-day values; add a date if you need timestamps.
- The arrival distribution is enforced by design; validate with a simple histogram if needed.
- If you rerun the notebook, arrivals may change due to randomness unless you set a random seed (e.g., np.random.seed(0)).
